# Describe GTDB Dataset
This notebook calculates the sampled proportion of each genome and extracts its strain-level taxonomy name from the original GTDB fasta files.

In [ ]:
import pandas as pd
import gzip
import os
from Bio import SeqIO
from tqdm.notebook import tqdm

In [ ]:
# --- CONFIGURATION ---
CSV_PATH = 'path/to/your/dataset.csv' # Path to the generated dataset CSV file
GTDB_DIR = '/path/to/GTDB/Bacteria'   # Path to the GTDB Bacteria directory containing .fna.gz files
OUTPUT_CSV = 'dataset_description.csv' # Output CSV file path


In [ ]:
print(f"Loading dataset from {CSV_PATH}...")
df = pd.read_csv(CSV_PATH)

# Calculate sampled length per genome
if 'genome_id' not in df.columns or 'seq' not in df.columns:
    raise ValueError("CSV must contain 'genome_id' and 'seq' columns.")
    
df['seq_len'] = df['seq'].apply(len)
sampled_lengths = df.groupby('genome_id')['seq_len'].sum().to_dict()
print(f"Found {len(sampled_lengths)} unique genomes in the dataset.")

In [ ]:
results = []

print("Processing original genome files to calculate proportions and extract taxonomy...")
for genome_id, sampled_len in tqdm(sampled_lengths.items()):
    file_path = os.path.join(GTDB_DIR, f"{genome_id}.fna.gz")
    
    if not os.path.exists(file_path):
        print(f"Warning: File {file_path} not found. Skipping {genome_id}...")
        continue
        
    original_len = 0
    tax_name = "Unknown"
    is_first_record = True
    
    try:
        with gzip.open(file_path, "rt") as f:
            for record in SeqIO.parse(f, "fasta"):
                if is_first_record:
                    # Extract the taxonomic/strain name from the FASTA header.
                    # BioPython stores the full header string after the first space in 'record.description'
                    # For example: ">RS_GCF_000005845.2 Escherichia coli str. K-12 substr. MG1655"
                    description = record.description
                    if description.startswith(record.id):
                        tax_name = description[len(record.id):].strip()
                    else:
                        tax_name = description.strip()
                    is_first_record = False
                    
                original_len += len(record.seq)
                
        proportion = sampled_len / original_len if original_len > 0 else 0
        
        results.append({
            "GTDB_ID": genome_id,
            "Tax_Name": tax_name,
            "Sampled_Proportion": proportion
        })
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

In [ ]:
out_df = pd.DataFrame(results)
out_df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSuccessfully processed {len(out_df)} genomes.")
print(f"Dataset description saved to {OUTPUT_CSV}")
out_df.head()